# Session 1 · Loading data

Companion notebook for the **Loading data** section of the slides.

| Part | What we see |
|---|---|
| 1 | One small file hides three loading traps |
| 2 | Predict, then run four loads of `orders.tsv` |
| 3 | Only the dtype mistake fails silently |
| 4 | `read_excel` and `sheet_name` |
| 5 | `json_normalize` and `record_path` |

All files are in the `data/` folder next to this notebook.

In [4]:
# Excel support needs openpyxl. Run once if it is missing:
# %pip install openpyxl

import json
import pandas as pd

pd.__version__

'3.0.6'

## 1 · One small file hides three loading traps

`orders.tsv` looks harmless: 3 orders, 6 columns. Before loading it with pandas,
let's look at the **raw bytes** of the first two lines.

In [2]:
f = "data/orders.tsv"

with open(f, "rb") as fh:
    raw = fh.read()

raw[:80]

b'order_id\tcustomer\tcountry\torder_date\ttotal\tstatus\n00417\tH\xe9l\xe8ne Dubois\tFrance\t202'

Three things to notice in the bytes:

- **Tabs** (`\t`) separate the columns, not commas.
- **Accents**: `H\xe9l\xe8ne` is *Hélène* saved in **latin-1**. In UTF-8, `é` would be two bytes (`\xc3\xa9`).
- **Leading zeros**: `00417` is an identifier, not a number.

> **Goal:** after loading, `order_id` must still read `00417`, and every accent must be correct.

## 2 · Predict what each of the four loads returns

Before running each cell, write down:

| Load | Error? | Shape | dtype of `order_id` |
|---|---|---|---|
| `df1` | | | |
| `df2` | | | |
| `df3` | | | |
| `df4` | | | |

The two loads that may fail are wrapped in `try/except`, so the notebook keeps running and still prints the error.

### Load 1: all defaults

In [3]:
f = "data/orders.tsv"

try:
    df1 = pd.read_csv(f)
except Exception as e:
    print(type(e).__name__, "->", e)

UnicodeDecodeError -> 'utf-8' codec can't decode byte 0xe9 in position 57: invalid continuation byte


### Load 2: fix the separator only

In [4]:
try:
    df2 = pd.read_csv(f, sep="\t")
except Exception as e:
    print(type(e).__name__, "->", e)

UnicodeDecodeError -> 'utf-8' codec can't decode byte 0xe9 in position 57: invalid continuation byte


Same error: pandas must **decode** the bytes into text before it can split on tabs,
so the separator never gets a chance.

### Load 3: separator and encoding

In [5]:
df3 = pd.read_csv(f, sep="\t", encoding="latin-1")
print(df3.shape)
df3

(3, 6)


,order_id,customer,country,order_date,total,status
0,417,Hélène Dubois,France,2026-10-03,129.9,shipped
1,418,Marco Rossi,Spain,2026-10-04,45.0,delivered
2,419,Café Lumière,Germany,2026-10-05,NaN,pending


In [6]:
df3.dtypes

order_id        int64
customer          str
country           str
order_date        str
total         float64
status            str
dtype: object

Accents are correct, but `order_id` became **417**: pandas saw digits and guessed a number.
No error, no warning.

### Load 4: separator, encoding and dtype

In [7]:
df4 = pd.read_csv(f, sep="\t", encoding="latin-1",
                  dtype={"order_id": str})
print(df4.shape)
df4

(3, 6)


,order_id,customer,country,order_date,total,status
0,00417,Hélène Dubois,France,2026-10-03,129.9,shipped
1,00418,Marco Rossi,Spain,2026-10-04,45.0,delivered
2,00419,Café Lumière,Germany,2026-10-05,NaN,pending


In [8]:
df4.dtypes

order_id          str
customer          str
country           str
order_date        str
total         float64
status            str
dtype: object

`'00417'` is kept as text!

In [9]:
df4["order_date"] = pd.to_datetime(
    df4["order_date"],
    format="%Y-%m-%d",
    errors="coerce", 
)

df4.dtypes

order_id                 str
customer                 str
country                  str
order_date    datetime64[us]
total                float64
status                   str
dtype: object

In [10]:
df4

,order_id,customer,country,order_date,total,status
0,00417,Hélène Dubois,France,2026-10-03,129.9,shipped
1,00418,Marco Rossi,Spain,2026-10-04,45.0,delivered
2,00419,Café Lumière,Germany,2026-10-05,NaN,pending


### Variant: fix only the encoding

What happens if we get the encoding right but forget `sep`?

In [11]:
df_variant = pd.read_csv(f, encoding="latin-1")
print(df_variant.shape)
df_variant

(3, 1)


,order_id\tcustomer\tcountry\torder_date\ttotal\tstatus
0,00417\tHélène Dubois\tFrance\t2026-10-03\t129....
1,00418\tMarco Rossi\tSpain\t2026-10-04\t45.00\t...
2,00419\tCafé Lumière\tGermany\t2026-10-05\t\tpe...


In [14]:
df_variant.columns.to_list()

['order_id\tcustomer\tcountry\torder_date\ttotal\tstatus']

One single column whose name and values still contain the tabs (`\t`).

## Only the dtype mistake fails silently

| Mistake | What you see | How loud |
|---|---|---|
| wrong `sep` | shape `(3, 1)` | **visible**: `df.shape` gives it away |
| wrong `encoding` | `UnicodeDecodeError` | **loud**: the load stops |
| missing `dtype` | `00417` becomes `417` | **silent**: no error, no warning |

Rule of thumb: pass `dtype=str` for ids, zip codes and phone numbers.

## 4 · `sheet_name` selects one sheet or all of them

`orders.xlsx` has two sheets: **Orders** and **Refunds**.

In [15]:
fx = "data/orders.xlsx"

pd.read_excel(fx)          # default: the first sheet

,order_id,customer,country,order_date,total,status
0,417,Hélène Dubois,France,2026-10-03,129.9,shipped
1,418,Marco Rossi,Spain,2026-10-04,45.0,delivered
2,419,Café Lumière,Germany,2026-10-05,NaN,pending


In [16]:
pd.read_excel(fx, sheet_name="Refunds")   # one sheet by name

,order_id,refund_date,amount,reason
0,417,2026-10-10,29.9,damaged item
1,418,2026-10-12,45.0,wrong size


In [18]:
sheets = pd.read_excel(fx, sheet_name=None)   # all sheets
type(sheets), list(sheets.keys())
sheets["Orders"]

,order_id,customer,country,order_date,total,status
0,417,Hélène Dubois,France,2026-10-03,129.9,shipped
1,418,Marco Rossi,Spain,2026-10-04,45.0,delivered
2,419,Café Lumière,Germany,2026-10-05,NaN,pending


In [19]:
# A dict {sheet name: DataFrame}: loop over it like any dict
for name, sheet in sheets.items():
    print(f"{name}: {sheet.shape}")

Orders: (3, 6)
Refunds: (2, 4)


After loading, the DataFrame API is identical to the CSV case, **including the silent trap**:
look at `order_id` above. The Excel file stores `'00417'` as text, yet pandas shows `417`.
The same `dtype` habit applies:

In [20]:
orders = pd.read_excel(fx, sheet_name="Orders", dtype={"order_id": str})
orders.dtypes

order_id                 str
customer                 str
country                  str
order_date    datetime64[us]
total                float64
status                   str
dtype: object

In [21]:
orders

,order_id,customer,country,order_date,total,status
0,00417,Hélène Dubois,France,2026-10-03,129.9,shipped
1,00418,Marco Rossi,Spain,2026-10-04,45.0,delivered
2,00419,Café Lumière,Germany,2026-10-05,NaN,pending


## 5 · `json_normalize` flattens nested dictionaries

Each order in `orders.json` has a nested `customer` **dict** and a nested list of `items`.

In [22]:
with open("data/orders.json", encoding="utf-8") as fh:
    data = json.load(fh)

data[0]

{'order_id': '00417',
 'customer': {'name': 'Hana', 'country': 'France'},
 'items': [{'sku': 'A1', 'qty': 2}, {'sku': 'B2', 'qty': 1}]}

`pd.DataFrame` keeps the nesting: whole dicts and lists end up inside cells.

In [23]:
pd.DataFrame(data)

,order_id,customer,items
0,00417,"{'name': 'Hana', 'country': 'France'}","[{'sku': 'A1', 'qty': 2}, {'sku': 'B2', 'qty':..."
1,00418,"{'name': 'Lucía', 'country': 'Spain'}","[{'sku': 'C3', 'qty': 5}]"


`json_normalize` flattens nested **dicts** into dotted column names.
Nested **lists** stay unexpanded.

In [24]:
flat = pd.json_normalize(data)
flat

,order_id,items,customer.name,customer.country
0,00417,"[{'sku': 'A1', 'qty': 2}, {'sku': 'B2', 'qty':...",Hana,France
1,00418,"[{'sku': 'C3', 'qty': 5}]",Lucía,Spain


In [25]:
flat.columns.tolist()

['order_id', 'items', 'customer.name', 'customer.country']

### `record_path` gives one row per nested item

- `record_path="items"`: one row per item
- `meta=[...]`: parent fields copied onto every row (use a list for nested keys)

In [26]:
items = pd.json_normalize(data, record_path="items",
                          meta=["order_id", ["customer", "country"], ["customer", "name"]],
                          sep='_')
items

,sku,qty,order_id,customer_country,customer_name
0,A1,2,00417,France,Hana
1,B2,1,00417,France,Hana
2,C3,5,00418,Spain,Lucía


Order 00417 has two items, order 00418 has one: 3 rows in total.
`order_id` stays `'00417'` because JSON stores it as a string.

## 6 · Profile before you change anything

Six calls, always in the same order, on every new dataset. Here on `df4`, the correct load of `orders.tsv`.

| Call | Question it answers |
|---|---|
| `.shape` | How many rows and columns? |
| `.dtypes` | Which type is each column? |
| `.head(10)` | What do real rows look like? |
| `.info()` | Where are the NULLs? How much memory? |
| `.describe()` | Which range do numbers span? |
| `.value_counts()` | Which is the number of unique values? |

In [27]:
df4.shape

(3, 6)

In [28]:
df4.dtypes

order_id                 str
customer                 str
country                  str
order_date    datetime64[us]
total                float64
status                   str
dtype: object

In [29]:
df4.head(10)

,order_id,customer,country,order_date,total,status
0,00417,Hélène Dubois,France,2026-10-03,129.9,shipped
1,00418,Marco Rossi,Spain,2026-10-04,45.0,delivered
2,00419,Café Lumière,Germany,2026-10-05,NaN,pending


In [30]:
df4.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    3 non-null      str           
 1   customer    3 non-null      str           
 2   country     3 non-null      str           
 3   order_date  3 non-null      datetime64[us]
 4   total       2 non-null      float64       
 5   status      3 non-null      str           
dtypes: datetime64[us](1), float64(1), str(4)
memory usage: 276.0 bytes


In [31]:
df4.describe()

,order_date,total
count,3,2.000000
mean,2026-10-04 00:00:00,87.450000
min,2026-10-03 00:00:00,45.000000
25%,2026-10-03 12:00:00,66.225000
50%,2026-10-04 00:00:00,87.450000
75%,2026-10-04 12:00:00,108.675000
max,2026-10-05 00:00:00,129.900000
std,NaN,60.033366


In [32]:
df4["status"].value_counts()

status
shipped      1
delivered    1
pending      1
Name: count, dtype: int64

## Practice: profile three files, answer five questions

Three **real** datasets, one per format. For each one you answer the five questions from the slide:

1. How many rows and columns?
2. Which columns have NULLs, and how many?
3. Which columns need a different dtype?
4. What are the unique `status` values?
5. What is the date range?

| File | What it is | Status column | Date column |
|---|---|---|---|
| `data/velib_stations.csv` | every Vélib bike-sharing station in Greater Paris | `is_installed` | `duedate` |
| `data/paris_roadworks.xlsx` | roadworks that disrupt traffic in Paris | `statut` | `date_debut`, `date_fin` |
| `data/pandas_issues.json` | the 100 latest issues and pull requests of the pandas project on GitHub | `state` | `created_at` |

The questions get harder as you go. Write your code in the empty cell below each question.

**Q1.** Open `data/velib_stations.csv` in binary mode and print its first 150 bytes.
Which character separates the columns?

In [ ]:
# Your code here

In [2]:
# SOLUTION
with open("data/velib_stations.csv", "rb") as fh:
    print(fh.read(150))
# The columns are separated by ";" (semicolon), not by commas.

b'\xef\xbb\xbfstationcode;name;is_installed;capacity;numdocksavailable;numbikesavailable;mechanical;ebike;is_renting;is_returning;duedate;coordonnees_geo;nom_arr'


**Q2.** Load the file into a DataFrame called `velib`. How many rows and columns does it have?

In [5]:
# Your code here

In [6]:
# SOLUTION
velib = pd.read_csv("data/velib_stations.csv", sep=";")
velib.shape          # (1520, 15)

(1520, 15)

**Q3.** Which columns have NULLs, and how many?

In [ ]:
# Your code here

In [8]:
# SOLUTION
velib.info()
# Only station_opening_hours has NULLs: 1520 out of 1520, the column is empty.

<class 'pandas.DataFrame'>
RangeIndex: 1520 entries, 0 to 1519
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   stationcode                  1520 non-null   int64  
 1   name                         1520 non-null   str    
 2   is_installed                 1520 non-null   str    
 3   capacity                     1520 non-null   int64  
 4   numdocksavailable            1520 non-null   int64  
 5   numbikesavailable            1520 non-null   int64  
 6   mechanical                   1520 non-null   int64  
 7   ebike                        1520 non-null   int64  
 8   is_renting                   1520 non-null   str    
 9   is_returning                 1520 non-null   str    
 10  duedate                      1520 non-null   str    
 11  coordonnees_geo              1520 non-null   str    
 12  nom_arrondissement_communes  1520 non-null   str    
 13  code_insee_commune           

**Q4.** Look at `velib.dtypes` and `velib.head(10)`. Which columns need a different dtype?
Then reload the file so that the two identifier columns, `stationcode` and `code_insee_commune`, are kept as text.

In [ ]:
# Your code here

In [9]:
# SOLUTION
print(velib.dtypes)
display(velib.head(10))
# stationcode, code_insee_commune : identifiers read as int64 -> should be text
# duedate                         : a date stored as text      -> should be a datetime
# coordonnees_geo                 : latitude and longitude glued in one text column
# station_opening_hours           : float64 only because it is entirely empty

velib = pd.read_csv("data/velib_stations.csv", sep=";",
                    dtype={"stationcode": str, "code_insee_commune": str})
velib.dtypes

stationcode                      int64
name                               str
is_installed                       str
capacity                         int64
numdocksavailable                int64
numbikesavailable                int64
mechanical                       int64
ebike                            int64
is_renting                         str
is_returning                       str
duedate                            str
coordonnees_geo                    str
nom_arrondissement_communes        str
code_insee_commune               int64
station_opening_hours          float64
dtype: object


,stationcode,name,is_installed,capacity,numdocksavailable,numbikesavailable,mechanical,ebike,is_renting,is_returning,duedate,coordonnees_geo,nom_arrondissement_communes,code_insee_commune,station_opening_hours
0,40001,Hôpital Mondor,OUI,28,14,13,9,4,OUI,OUI,2026-11-07T12:49:09+00:00,"48.798922410229, 2.4537451531298",Créteil,94028,NaN
1,32304,Charcot - Benfleet,OUI,28,7,20,7,13,OUI,OUI,2026-11-07T12:47:58+00:00,"48.878370277021, 2.440523876268",Romainville,93063,NaN
2,9020,Toudouze - Clauzel,OUI,21,16,5,0,5,OUI,OUI,2026-11-07T12:50:44+00:00,"48.87929591733507, 2.3373600840568547",Paris,75056,NaN
3,32017,Basilique,OUI,22,11,11,2,9,OUI,OUI,2026-11-07T12:42:08+00:00,"48.93626891059109, 2.3588666820200914",Saint-Denis,93066,NaN
4,11104,Charonne - Robert et Sonia Delaunay,OUI,20,20,0,0,0,OUI,OUI,2026-11-07T12:50:36+00:00,"48.855907555969, 2.3925706744194",Paris,75056,NaN
5,5110,Lacépède - Monge,OUI,23,22,0,0,0,OUI,OUI,2026-11-07T12:46:50+00:00,"48.84389286531899, 2.3519663885235786",Paris,75056,NaN
6,6108,Saint-Romain - Cherche-Midi,OUI,17,3,13,3,10,OUI,OUI,2026-11-07T12:49:01+00:00,"48.84708159081946, 2.321374788880348",Paris,75056,NaN
7,33006,André Karman - République,OUI,31,26,4,3,1,OUI,OUI,2026-11-07T12:46:51+00:00,"48.91039875761846, 2.3851355910301213",Aubervilliers,93001,NaN
8,42016,Pierre et Marie Curie - Maurice Thorez,OUI,27,27,0,0,0,OUI,OUI,2026-11-07T12:50:44+00:00,"48.81580226360801, 2.376804985105991",Ivry-sur-Seine,94041,NaN
9,40011,Bas du Mont-Mesly,OUI,27,10,16,14,2,OUI,OUI,2026-11-07T12:45:55+00:00,"48.779751, 2.460837",Créteil,94028,NaN


stationcode                        str
name                               str
is_installed                       str
capacity                         int64
numdocksavailable                int64
numbikesavailable                int64
mechanical                       int64
ebike                            int64
is_renting                         str
is_returning                       str
duedate                            str
coordonnees_geo                    str
nom_arrondissement_communes        str
code_insee_commune                 str
station_opening_hours          float64
dtype: object

**Q5.** In this dataset the status column is `is_installed`. What are its unique values, and how many stations have each one?

In [ ]:
# Your code here

In [ ]:
# SOLUTION
velib["is_installed"].value_counts()
# OUI 1517, NON 3

**Q6.** How many sheets does `data/paris_roadworks.xlsx` contain, and what are their names?

In [ ]:
# Your code here

In [ ]:
# SOLUTION
sheets = pd.read_excel("data/paris_roadworks.xlsx", sheet_name=None)
len(sheets), list(sheets.keys())     # (1, ['Sheet1'])

**Q7.** Load the sheet into a DataFrame called `works`. How many rows and columns does it have?

In [ ]:
# Your code here

In [ ]:
# SOLUTION
works = pd.read_excel("data/paris_roadworks.xlsx")
works.shape          # (88, 11)

**Q8.** Look at `works.dtypes` and `works.head(10)`. What happened to the postcode column `cp_arrondissement`?
Reload the file so that it is kept as text. Which other columns should not be text?

In [ ]:
# Your code here

In [ ]:
# SOLUTION
print(works.dtypes)
display(works.head(10))
# cp_arrondissement is float64: 75001 became 75001.0 (a number with a decimal part).
# date_debut and date_fin are dates stored as text.

works = pd.read_excel("data/paris_roadworks.xlsx",
                      dtype={"cp_arrondissement": str})
works["cp_arrondissement"].head()

**Q9.** Load `data/pandas_issues.json` with the `json` module into a variable called `raw`.
How many records does it contain? Print the first one: which fields are nested?

In [ ]:
# Your code here

In [ ]:
# SOLUTION
with open("data/pandas_issues.json", encoding="utf-8") as fh:
    raw = json.load(fh)

print(len(raw))      # 100
raw[0]
# user is a nested dict; labels is a nested list of dicts.

**Q10.** Flatten the records with `json_normalize` into a DataFrame called `issues`.
How many rows and columns does it have? Which columns come from the nested dict?

In [ ]:
# Your code here

In [ ]:
# SOLUTION
issues = pd.json_normalize(raw)
print(issues.shape)  # (100, 10)
issues.columns.tolist()
# user.login and user.type come from the nested user dict;
# labels is still a list in each cell.

**Q11.** What are the unique values of `state`, and how many records have each one?
How many of the records are pull requests?

In [ ]:
# Your code here

In [ ]:
# SOLUTION
print(issues["state"].value_counts())        # open 52, closed 48
issues["is_pull_request"].value_counts()     # True 89, False 11

**Q12.** Build a table with **one row per label**, keeping the issue `number` and `state` on every row. How many rows does it have? Which label is the most frequent?

In [ ]:
# Your code here

In [ ]:
# SOLUTION
labels = pd.json_normalize(raw, record_path="labels",
                           meta=["number", "state"])
print(labels.shape)                          # (191, 3)
labels["name"].value_counts().head()
# Bug is the most frequent label (33 rows).